In [12]:
import pandas as pd 
import numpy as np
import cv2 
import os 
import matplotlib.pyplot as plt

# Train model DBnet nhận diện chữ

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

In [ ]:
# ==================== 1. CẤU HÌNH THAM SỐ & ĐƯỜNG DẪN ====================
IMAGE_DIR = r"../Data/vietnamese_original/train_images"      # Thư mục chứa ảnh train
LABEL_DIR = r"../Data/vietnamese_original/labels"            # Thư mục chứa file txt nhãn (vd: gt001.txt)
TEST_IMAGE_DIR = r"../Data/vietnamese_original/test_image"   # Thư mục chứa ảnh cần test
TEST_RESULT_DIR = r"../Data/test_results"                    # Thư mục lưu kết quả dự đoán

BATCH_SIZE = 2                 
EPOCHS = 5                     
LEARNING_RATE = 1e-3           
IMG_SIZE = (512, 512)          

# ==================== 2. HÀM ÁNH XẠ TÊN FILE (img001 -> gt001) ====================
def get_label_path(img_filename, label_dir):
    base_name, _ = os.path.splitext(img_filename)
    # Thay thế tiền tố 'img' thành 'gt' nếu tên file của bạn theo quy tắc này
    label_base = base_name.replace("img", "gt")
    return os.path.join(label_dir, label_base + ".txt")

# ==================== 3. LOAD DỮ LIỆU TỪ THƯ MỤC ====================
class FolderTextDataset(Dataset):
    def __init__(self, image_dir, label_dir, img_size):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.img_size = img_size
        
        valid_exts = ('.png', '.jpg', '.jpeg', '.bmp')
        self.image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(valid_exts)]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)
        label_path = get_label_path(img_name, self.label_dir)

        image = cv2.imread(img_path)
        if image is None:
            image = np.zeros((*self.img_size, 3), dtype=np.uint8)

        h, w = image.shape[:2]
        mask = np.zeros((h, w), dtype=np.float32)
        
        if os.path.exists(label_path):
            with open(label_path, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split(',')
                    if len(parts) >= 9:
                        coords = [float(p) for p in parts[:8]]
                        pts = np.array(coords, dtype=np.int32).reshape(4, 2)
                        cv2.fillPoly(mask, [pts], 1.0)

        image = cv2.resize(image, self.img_size)
        mask = cv2.resize(mask, self.img_size, interpolation=cv2.INTER_NEAREST)

        img_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()
        
        return img_tensor, mask_tensor

# ==================== 4. KHỞI TẠO MÔ HÌNH (ResNet18 Backbone) ====================
class QuickTextDetector(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Bỏ 2 lớp cuối của ResNet18 (avgpool và fc)
        self.features = nn.Sequential(*list(backbone.children())[:-2])
        
        self.head = nn.Sequential(
            nn.Conv2d(512, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 1, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        input_size = x.shape[2:]       # Lấy kích thước gốc (ví dụ: 512, 512)
        x = self.features(x)           # Qua backbone
        x = self.head(x)               # Qua head
        # Khôi phục về kích thước chuẩn để khớp với mask mục tiêu
        x = F.interpolate(x, size=input_size, mode='bilinear', align_corners=False)
        return x

# ==================== 5. HUẤN LUYỆN (TRAIN) ====================
def train_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Đang huấn luyện trên thiết bị: {device}")

    dataset = FolderTextDataset(IMAGE_DIR, LABEL_DIR, IMG_SIZE)
    if len(dataset) == 0:
        print("Cảnh báo: Không tìm thấy file ảnh train nào trong IMAGE_DIR!")
        return None
        
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    model = QuickTextDetector().to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            preds = model(images)
            loss = criterion(preds, masks)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {total_loss/len(dataloader):.4f}")

    torch.save(model.state_dict(), "text_detector_model.pth")
    print("Đã train xong và lưu model tại: text_detector_model.pth")
    return model

# ==================== 6. KIỂM THỬ (TEST / INFERENCE) ====================
def test_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = QuickTextDetector().to(device)
    
    model_path = "text_detector_model.pth"
    if not os.path.exists(model_path):
        print(f"Không tìm thấy file weights {model_path}, vui lòng train trước!")
        return
        
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    os.makedirs(TEST_RESULT_DIR, exist_ok=True)
    valid_exts = ('.png', '.jpg', '.jpeg', '.bmp')
    
    if not os.path.exists(TEST_IMAGE_DIR):
        print(f"Thư mục test '{TEST_IMAGE_DIR}' không tồn tại!")
        return
        
    test_files = [f for f in os.listdir(TEST_IMAGE_DIR) if f.lower().endswith(valid_exts)]
    if len(test_files) == 0:
        print(f"Không tìm thấy ảnh nào trong thư mục test: {TEST_IMAGE_DIR}")
        return

    print(f"Bắt đầu kiểm thử {len(test_files)} ảnh...")
    with torch.no_grad():
        for img_name in test_files:
            img_path = os.path.join(TEST_IMAGE_DIR, img_name)
            image = cv2.imread(img_path)
            if image is None:
                continue

            h, w = image.shape[:2]
            resized_img = cv2.resize(image, IMG_SIZE)
            
            # Chuẩn hóa tensor
            img_tensor = torch.from_numpy(resized_img).permute(2, 0, 1).float().unsqueeze(0) / 255.0
            img_tensor = img_tensor.to(device)
            
            # Dự đoán
            pred = model(img_tensor)
            pred_mask = pred.squeeze().cpu().numpy() # [H, W]
            
            # Ngưỡng hóa (Threshold > 0.5) và khôi phục về kích thước gốc của ảnh test
            pred_mask = (pred_mask > 0.5).astype(np.uint8) * 255
            pred_mask = cv2.resize(pred_mask, (w, h), interpolation=cv2.INTER_NEAREST)
            
            # Lưu ảnh kết quả mask
            save_path = os.path.join(TEST_RESULT_DIR, "pred_" + img_name)
            cv2.imwrite(save_path, pred_mask)
            print(f"Đã lưu kết quả test: {save_path}")

if __name__ == "__main__":
    # 1. Chạy train
    train_model()
    
    # 2. Chạy test
    test_model()

Đang huấn luyện trên thiết bị: cuda
Epoch [1/5] - Loss: 0.0045
Epoch [2/5] - Loss: 0.0000
Epoch [3/5] - Loss: 0.0000
Epoch [4/5] - Loss: 0.0000


KeyboardInterrupt: 